# TP2 — Monitoring d'API + démo finale du système
## Fil rouge : Churn Predictor (Session 5)

**Durée estimée : 1h05**

### 🎯 Objectifs
- Comprendre comment **étendre une API FastAPI** avec du monitoring (logging, metrics, drift-check).
- Simuler un **trafic réaliste** : normal puis drift, voir le système réagir.
- Élaborer une **stratégie de retraining** Tech Lead.
- Terminer le module avec une **démo end-to-end** du système complet.

### 📦 Pré-requis
- Le dossier `S5_churn_api_extended_solution/` (ou votre version étendue) est dézippé sur votre poste.
- Ce notebook est exécuté **à la racine** de ce repo.
- Le repo contient `artifacts/best_model.joblib`, `manifest.json`, `baseline_train.csv`.


## 1. Architecture du monitoring — vue d'ensemble

```
                  ┌──────────────┐
   client ─────▶  │   /predict   │  ──▶ predict + log to JSONL
                  └──────────────┘
                         │
                         ▼
                  monitoring/predictions.log.jsonl
                         │
                         ▼
                  ┌──────────────┐
                  │ /drift-check │  ──▶ read recent logs,
                  └──────────────┘      compute PSI vs baseline
```

**Choix de design retenus** :
- **JSONL append-only** plutôt que base SQL : simple, robuste, parsable streamingement.
- **Baseline = snapshot des features train** (`baseline_train.csv`) : référence figée.
- **PSI sur 6 features critiques** seulement (3 num + 3 cat) : monitoring ≠ rétraining, on observe l'essentiel.
- **Endpoint `/drift-check` synchrone** : OK pour 1k preds; au-delà, partir sur un job batch nightly.

### Modules ajoutés au repo

| Fichier | Rôle |
|---------|------|
| `app/monitoring.py` | log_prediction, read_recent_logs, psi_*, compute_drift, status_from_max_psi |
| `app/main.py` (modif) | `/metrics`, `/drift-check`, appel `log_prediction` dans `/predict` |
| `app/schemas.py` (modif) | `MetricsResponse`, `DriftCheckResponse` |
| `app/config.py` (modif) | `LOG_PATH`, `BASELINE_TRAIN_PATH`, `PSI_*_THRESHOLD` |


## 2. Connexion à l'API étendue via TestClient

In [ ]:
import sys
import os
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "S5_churn_api_extended_squelette" / "S5_churn_api_extended_squelette"))

# Clean any pre-existing log
log_file = Path("monitoring/predictions.log.jsonl")
if log_file.exists():
    log_file.unlink()
    print(f"Removed existing {log_file}")

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()
print("✅ TestClient ready")

In [ ]:
# Verify the new monitoring endpoints exist
schema = client.get("/openapi.json").json()
print("Endpoints:")
for path, methods in schema["paths"].items():
    for method in methods:
        if method in ("get", "post"):
            tags = methods[method].get("tags", [])
            print(f"  {method.upper():6s} {path}  {tags}")


## 3. Phase 1 — Trafic normal (50 requêtes du test set)

On envoie 50 prédictions tirées du **test set** (donc même distribution que le train) → le drift devrait rester faible.

In [ ]:
import pandas as pd
import random

X_test_fe = pd.read_csv("artifacts/baseline_train.csv")  # use the available CSV as our 'normal' source

# Drop the engineered features (the API expects raw features only)
ENGINEERED_COLS = ["tenure_group", "services_count", "has_internet", "avg_charge_per_month"]
RAW_COLS = [c for c in X_test_fe.columns if c not in ENGINEERED_COLS]

random.seed(0)
sample_indices = random.sample(range(len(X_test_fe)), 50)

ok_count = 0
for idx in sample_indices:
    row = X_test_fe.iloc[idx][RAW_COLS]
    payload = {
        k: (int(v) if isinstance(v, (int,)) or hasattr(v, 'item') and isinstance(v.item(), int)
            else (float(v) if isinstance(v, (float,)) or 'float' in str(type(v))
                  else str(v)))
        for k, v in row.items()
    }
    # cast cleanly
    for k in ["SeniorCitizen", "tenure"]:
        payload[k] = int(payload[k])
    for k in ["MonthlyCharges", "TotalCharges"]:
        payload[k] = float(payload[k])
    
    r = client.post("/predict", json=payload)
    if r.status_code == 200:
        ok_count += 1

print(f"Sent 50 normal requests, {ok_count} accepted")


In [ ]:
# Check metrics
r = client.get("/metrics")
print("Metrics after 50 normal requests:")
print(json.dumps(r.json(), indent=2))

# Check drift
r = client.get("/drift-check")
print("\nDrift check after 50 normal requests:")
print(json.dumps(r.json(), indent=2))


💡 **Lecture attendue** : `status: "ok"` (ou éventuellement `warning` léger). PSI tous < 0.10. C'est notre **état de référence** : système healthy.

## 4. Phase 2 — Trafic drifté (50 requêtes simulant un changement marketing)

On simule maintenant 50 requêtes où **`MonthlyCharges` est multiplié par 1.30** et `Contract` est forcé à `Month-to-month` à 70 % — comme si le marketing avait changé la grille tarifaire.

In [ ]:
ok_count = 0
for idx in sample_indices:
    row = X_test_fe.iloc[idx][RAW_COLS].copy()
    row["MonthlyCharges"] = float(row["MonthlyCharges"]) * 1.30
    if random.random() < 0.7:
        row["Contract"] = "Month-to-month"
    
    payload = {k: v for k, v in row.items()}
    for k in ["SeniorCitizen", "tenure"]:
        payload[k] = int(payload[k])
    for k in ["MonthlyCharges", "TotalCharges"]:
        payload[k] = float(payload[k])
    for k, v in payload.items():
        if not isinstance(v, (int, float)):
            payload[k] = str(v)
    
    r = client.post("/predict", json=payload)
    if r.status_code == 200:
        ok_count += 1

print(f"Sent 50 DRIFTED requests, {ok_count} accepted")


In [ ]:
r = client.get("/metrics")
print("Metrics after 50 normal + 50 drifted:")
print(json.dumps(r.json(), indent=2))

r = client.get("/drift-check")
print("\nDrift check (should now show drift):")
print(json.dumps(r.json(), indent=2))


🚨 **Lecture attendue** : `status: "warning"` ou `"critical"`. `MonthlyCharges` et `Contract` ont des PSI élevés. Le système **détecte le changement** sans qu'on lui dise rien.

🧠 **C'est le moment "ah oui, en fait ça marche"**. Le système qu'on a construit fait un travail réel : il vous prévient avant que vous découvriez par hasard qu'un dashboard business a chuté.

### En production, ce status alimenterait :
- **Slack/PagerDuty** : alerte à l'équipe ML
- **Dashboard Grafana** : visualisation continue
- **Auto-trigger d'un retraining job** (avec validation humaine)


## 5. Stratégie de retraining — la décision Tech Lead

🚨 **Le piège du retraining réflexe** : "drift détecté → retrainer". Trop simpliste.

### Les vraies questions à poser

| Question | Pourquoi |
|----------|----------|
| Le drift est-il **persistant** ou un pic temporaire ? | Promo flash 1 jour ≠ changement structurel |
| A-t-on **assez de données labellisées** post-drift ? | Sans labels, on ne sait pas si le modèle est encore bon |
| Le **score métier réel** (gain €) chute-t-il ? | Drift sur features ≠ baisse de performance garantie |
| Le coût du retraining (compute + revue) est-il **justifié** ? | Pas tous les drifts méritent un retrain |
| A-t-on **un plan de rollback** si le nouveau modèle est pire ? | Toujours |

### Le triggering arbre de décision

```
DRIFT DETECTED (PSI critique sur ≥1 feature)
    │
    ├─ Persistant > 7 jours ?
    │   ├─ Non → continuer monitoring, alerter
    │   └─ Oui → étape suivante
    │
    ├─ Score métier réel a chuté > 5% ?
    │   ├─ Non → adapter feature engineering / explorer
    │   └─ Oui → étape suivante
    │
    └─ Données labellisées disponibles ≥ 2 semaines ?
        ├─ Non → attendre + monitoring renforcé
        └─ Oui → LAUNCH RETRAINING (avec validation humaine)
```

### Cadence de retraining recommandée (sans drift)

| Fréquence | Cas d'usage |
|-----------|-------------|
| Hebdo | Retail, e-commerce, news (haute volatilité) |
| Mensuel | Banque, télécom, B2B SaaS |
| Trimestriel | Industrie, assurance long-terme |

🧠 **Le réflexe Tech Lead final** : un système ML en prod, c'est **20 % entraînement, 80 % opérations**. Cette session vous donne la dernière brique : **l'observabilité**.


## 6. Vérification finale : suite de tests étendue

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])


## 7. Démo finale du système complet — bouclage du module

🎬 **Le scénario** :

1. **Modèle** : entraîné en S1-S2 (HGBT tuné, seuil métier optimisé)
2. **Clusters** : explorés en S3, rejetés (gain non-significatif)
3. **API** : packagée en S4 (Pydantic, FastAPI, lifespan, tests)
4. **Monitoring** : ajouté en S5 (logging + drift detection)

Vous tenez maintenant un **système ML opérable de bout en bout**. C'est ce qu'on attend d'un Tech Lead.

### Récap de la chaîne

```
┌─────────────┐    ┌──────────┐    ┌────────────┐    ┌──────────────┐
│ Raw Telco   │───▶│ Training │───▶│ best_model │───▶│  FastAPI     │
│ data        │    │ Pipeline │    │   .joblib  │    │  + Pydantic  │
└─────────────┘    └──────────┘    └────────────┘    └──────────────┘
                                                            │
                                                            ▼
                                                    ┌──────────────┐
                                                    │ /predict     │
                                                    │ /metrics     │
                                                    │ /drift-check │
                                                    └──────────────┘
                                                            │
                                                    ┌───────┴───────┐
                                                    ▼               ▼
                                            JSONL log       PSI vs baseline
                                                            │
                                                            ▼
                                                    Alert if drift > 0.25
```


In [ ]:
client.__exit__(None, None, None)
print("TestClient closed.")


## 8. Bilan & au-revoir

### Compétences acquises sur le module (5 sessions × 21h)

#### Hard skills
- ✅ **EDA + preprocessing** propre, train/val/test stratifié
- ✅ **Pipelines scikit-learn** : ColumnTransformer + classifieur
- ✅ **Cross-validation** stratifiée multi-métriques
- ✅ **MLflow** : tracking d'expériences (params, metrics, artifacts)
- ✅ **GridSearchCV** rigoureux, syntax `clf__param`
- ✅ **Threshold tuning** par fonction de coût métier
- ✅ **Permutation importance** pour interpréter les modèles
- ✅ **Clustering non-supervisé** : k-means, DBSCAN, PCA, profiling
- ✅ **FastAPI** : Pydantic, endpoints, lifespan, tests
- ✅ **Drift detection** : PSI, KS test, alerting

#### Soft skills Tech Lead
- ✅ **Décision basée sur la donnée** : on rejette une feature qui n'aide pas
- ✅ **Discipline du test set** : une seule fois, à la fin
- ✅ **Reproductibilité** : tracking, manifest, versioning
- ✅ **Communication** : score business € plutôt que F1 abstrait
- ✅ **Anticipation prod** : monitoring dès le début, pas après le crash

### À remplir : votre auto-évaluation

> **Mes 3 plus gros takeaways du module** :
> 1. **La dérive en production est inévitable, la détecter tôt c'est survire** — Avoir construit le système PSI et vu comment MonthlyCharges × 1.30 déclenche des alertes me montre qu'on n'est jamais à l'abri. Monitoring = assurance pour le modèle.
> 2. **Un pipeline bien construit vaut mieux que 100 modèles compliqués** — ColumnTransformer + StandardScaler + modèle simple qui passe la validation en production, c'est déjà 80% de la valeur. Les 20% restants, c'est du tuning méticuleux.
> 3. **Le test set est sacré, on le touche une fois** — La tentation de l'overfitter est énorme avec GridSearchCV. Garder la discipline = résultats honnêtes et modèle réellement robuste.
>
> **Ce que je vais appliquer dès lundi** :
> - Intégrer du monitoring/logging dès le sprint 0, pas en phase finale en Rush.
> - Construire systématiquement un manifest (params, versions, dates) pour tracer l'origine de chaque modèle.
> - Communiquer en € ou impact métier (churn évité, revenu sauvé) plutôt qu'en F1-score quand j'explique aux stakeholders.
>
> **Ce que je n'ai pas encore maîtrisé et que je vais creuser** :
> - **MLflow Model Registry** pour versionner et servir les modèles en staging/prod sans redéployer le code.
> - **Feature Stores** (Feast, Tecton) pour centraliser les transformations et gérer le training/serving skew.
> - **A/B testing** on production : comment déployer un modèle new vs old en parallel et mesurer le vrai impact business.

### Pour aller plus loin

- **MLOps** approfondi : Kubeflow, MLflow Model Registry, feature stores (Feast, Tecton)
- **Explainability** : SHAP en profondeur, LIME, contrefactuels
- **Online learning** : adaptation continue, multi-armed bandits
- **Causal ML** : différencier corrélation et causalité (DoWhy, EconML)
- **Vector DB / RAG** : embeddings, search, génératif
- **LLMOps** : tracking des prompts, eval, monitoring spécifique aux LLM

---

🎓 **Bravo. Vous savez maintenant construire un produit ML de bout en bout.**

C'est plus que ce que beaucoup de "Data Scientists" en poste savent faire. Capitalisez là-dessus, et bon courage pour la suite. 👋